###**Examples to modify the prompt**

In [ ]:
url = "https://stergios.hopto.org:5020/get_answer"

In [ ]:
import requests

In [ ]:
data = {
    "query": "où était situé le bureau de J. de Siebenthal? ",
    "prompt": [{
                "role": "system",
                "content": "You are an expert in historical documents and you have access to the following documents: retrieved_docs"
            },
            {
                "role": "user",
                "content": "You are an expert in historical documents, answer the following question using the provided documents: user_question as a reminder, here are the retrieved_docs"
            }],
    "k": 10,
    "chroma_index": "apple_ocr_gpt_correction_openai_embeddings" # can be any of ["apple_ocr_openai", "apple_ocr_gpt_correction_openai_embeddings"]. default: apple_ocr_openai
}

In [ ]:
requests.post(url, json=data).text

In [ ]:
# validation test
# retrieved_docs and user_question are reserved and will be replaced with the corresponding values
data = {
    "query": "Who was garibaldi",
    "prompt": [{
                "role": "system",
                "content": "You are an expert in historical documents and you have access to the following documents: retrieved_docs. Ensure to start and end every answer with the word 'banana'"
            },
            {
                "role": "user",
                "content": "You are an expert in historical documents, answer the following question using the provided documents: user_question as a reminder, here are the retrieved_docs"
            }],
    "k": 30,
    "chroma_index": "apple_ocr_gpt_correction_openai_embeddings" # can be any of ["apple_ocr_openai", "apple_ocr_gpt_correction_openai_embeddings"]. default: apple_ocr_openai

}
requests.post(url, json=data).text

### **Download index and do BM25 on it**

In [ ]:
import requests
from tqdm.auto import tqdm # Import tqdm for progress bar

url = "https://stergios.hopto.org:5020/download_index"
filename = "download_index.zip"

try:
    response = requests.get(url, stream=True)
    response.raise_for_status()  # Raise an exception for HTTP errors

    # Get total file size from headers, if available
    total_size = int(response.headers.get('content-length', 0))

    with open(filename, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=filename) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
    print(f"Successfully downloaded {filename}")
except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")

In [ ]:
import zipfile
import os

extract_dir = "faiss_index"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile("download_index.zip", 'r') as z:
    z.extractall(extract_dir)

print("Extracted files:", os.listdir(extract_dir))

### **Testing first**

In [ ]:
pip install chromadb

In [ ]:
from chromadb import PersistentClient
import chromadb

chromadb.api.client.SharedSystemClient.clear_system_cache()

client = PersistentClient(
    path="faiss_index/apple_ocr_openai"
)

collections = client.list_collections()
print(collections)

In [ ]:
collection = client.get_collection(name="apple_ocr_openai")

In [ ]:
data = collection.get()
print(data.keys())

In [ ]:
documents = data["documents"]
doc_ids = data["ids"]

print("Docs:", len(documents))
print("First doc preview:\n", documents[0][:300])


In [ ]:
!pip install rank-bm25

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

In [ ]:
query = "Qui est Jules Claretie et quel est son rapport à la litterature?"
scores = bm25.get_scores(query.lower().split())

In [ ]:
import numpy as np

k = 5
top_idx = np.argsort(scores)[::-1][:k]

In [ ]:
for rank, i in enumerate(top_idx, 1):
    print(f"\nRank {rank} | Score: {scores[i]:.2f}")
    print(documents[i][:300])

## BM25 experiment

Goal of experiment is to generate top k using BM25, then prompt LLM with question list (131) and compare results with RAG system

This can be done with apple ocr open ai and apple ocr gpt with correction (check params)

**Import CSV evaluation_questions from git**

#### **Helper**

In [ ]:
def load_documents_from_chroma(persist_dir, collection_name):
    print("📂 Loading Chroma DB from:", persist_dir)

    import chromadb
    from chromadb import PersistentClient

    chromadb.api.client.SharedSystemClient.clear_system_cache()

    client = PersistentClient(path=persist_dir)
    collection = client.get_collection(name=collection_name)

    data = collection.get()
    documents = data["documents"]
    doc_ids = data["ids"]

    print(f"✅ Loaded {len(documents)} documents")
    print("📄 First doc preview:", documents[0][:200])

    return documents, doc_ids

In [ ]:
from rank_bm25 import BM25Okapi

def build_bm25(documents):
    print("🔧 Building BM25 index...")

    tokenized_docs = [doc.lower().split() for doc in documents]
    bm25 = BM25Okapi(tokenized_docs)

    print("✅ BM25 index ready")
    return bm25

In [ ]:
import numpy as np

def bm25_retrieve(question, bm25, documents, doc_ids, top_k):
    print(f"\n🔍 BM25 retrieval | top_k={top_k}")
    print("❓ Question:", question)

    scores = bm25.get_scores(question.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]

    chunks = []
    for rank, i in enumerate(top_idx):
        print(f"  Rank {rank+1} | Score {scores[i]:.2f}")
        print("  Text preview:", documents[i][:150])

        chunks.append({
            "text": documents[i],
            "rank": rank,
            "score": float(scores[i]),
            "doc_id": doc_ids[i],
        })

    return chunks

In [ ]:
import os
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
from typing import List, Dict, Any

def generate_answer_openai(
    client: OpenAI,
    question: str,
    chunks: List[Dict],
    model: str = "gpt-4o",
) -> Dict[str, Any]:
    """
    Generate answer using OpenAI API with reranked chunks.
    """
    # Build context from reranked chunks
    context_text = "\n\n".join([
        f"Document {i+1}:\n{c.get('text', c.get('content', str(c)))}"
        for i, c in enumerate(chunks)
    ])



    messages = [
        {
            "role": "system",
            "content":
               "You are a retrieval-augmented question answering system. "
    "You must answer using only the provided context and no prior or external knowledge. "
    "You may combine information from multiple documents in the context to derive the answer. "
    "Do not introduce new facts, assumptions, or interpretations that are not supported by the context. "
    "If the answer cannot be clearly and directly derived from the provided context, "
    "respond exactly with: I don't know (for English questions) or je ne sais pas (for French questions). "
    "Your answers must be concise, factual, and limited to 1–2 sentences."
        },
        {
            "role": "user",
                  "content": (
            f"Question:\n{question}\n\n"
            f"Documents:\n{context_text}"
        ),

        }
    ]
    start_time = time.time()
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
        )
        latency_ms = (time.time() - start_time) * 1000

        return {
            "success": True,
            "answer": response.choices[0].message.content.strip(),
            "latency_ms": latency_ms,
        }
    except Exception as e:
        return {
            "success": False,
            "answer": "",
            "latency_ms": (time.time() - start_time) * 1000,
            "error": str(e),
        }

In [ ]:
from openai import OpenAI

def run_single_bm25_rag_test():
    # ---- CONFIG ----
    persist_dir = "faiss_index/apple_ocr_openai"
    collection_name = "apple_ocr_openai"
    question = "Quand a été inauguré le canal de panama?"
    top_k = 3

    # ---- SETUP ----
    documents, doc_ids = load_documents_from_chroma(
        persist_dir,
        collection_name,
    )

    bm25 = build_bm25(documents)

    retrieved_chunks = bm25_retrieve(
        question,
        bm25,
        documents,
        doc_ids,
        top_k=top_k,
    )

    # ---- GENERATE ANSWER ----
    print("\n🧠 Generating answer with RAG...")
    openai_client = OpenAI()

    response = generate_answer_openai(
        client=openai_client,
        question=question,
        chunks=retrieved_chunks,
    )

    print("\n📝 FINAL ANSWER")
    print("Answer:", response["answer"])
    print("Latency (ms):", response["latency_ms"])

In [ ]:
import time
run_single_bm25_rag_test()

**Final helpers**

In [ ]:
import csv

def load_eval_dataset(csv_path):
    dataset = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            dataset.append({
                "question_id": f"q_{i:04d}",
                "question": row["question"],
                "groundtruth": row["groundtruth"],
                "category": row.get("category"),
            })
    print(f"✅ Loaded {len(dataset)} evaluation questions")
    return dataset


In [ ]:
import json
from openai import OpenAI

def run_bm25_rag_eval(
    eval_csv_path,
    k,
    output_generations_path=None,
    output_gold_path="gold_answers.json",
):
    """
    Run BM25 + RAG evaluation for a given top-k.
    BM25 is rebuilt fresh on every call.
    """

    # ---- CONSTANT CONFIG (always the same) ----
    persist_dir = "faiss_index/apple_ocr_openai"
    collection_name = "apple_ocr_openai"  # internal Chroma collection name

    if output_generations_path is None:
        output_generations_path = f"generations_bm25_k{k}.jsonl"

    print(f"\n🚀 Running BM25–RAG evaluation | top_k={k}")

    # ---- LOAD EVAL DATA ----
    eval_data = load_eval_dataset(eval_csv_path)

    # ---- LOAD DOCUMENTS ----
    documents, doc_ids = load_documents_from_chroma(
        persist_dir,
        collection_name,
    )

    # ---- (RE)BUILD BM25 ----
    print("♻️ Rebuilding BM25 index...")
    bm25 = build_bm25(documents)

    openai_client = OpenAI()

    generations = []
    gold_answers = {}

    # ---- MAIN LOOP ----
    for item in eval_data:
        #print("\n==============================")
        #print("Question ID:", item["question_id"])
        #print("Question:", item["question"])

        retrieved_chunks = bm25_retrieve(
            item["question"],
            bm25,
            documents,
            doc_ids,
            top_k=k,
        )

        response = generate_answer_openai(
            client=openai_client,
            question=item["question"],
            chunks=retrieved_chunks,
        )

        generations.append({
            "question_id": item["question_id"],
            "question": item["question"],
            "generated_answer": response["answer"],
            "retrieved_chunks": [
                {
                    "text": c["text"],
                    "rank": c["rank"],
                    "score": c["score"],
                }
                for c in retrieved_chunks
            ],
            "method_metadata": {
                "retriever": "bm25",
                "top_k": k,
            },
        })

        gold_answers[item["question_id"]] = item["groundtruth"]

    # ---- WRITE FILES ----
    with open(output_generations_path, "w", encoding="utf-8") as f:
        for g in generations:
            f.write(json.dumps(g, ensure_ascii=False) + "\n")

    with open(output_gold_path, "w", encoding="utf-8") as f:
        json.dump(gold_answers, f, ensure_ascii=False, indent=2)

    print("\n✅ Saved:")
    print(" -", output_generations_path)
    print(" -", output_gold_path)


### Experiment 1:

For the following experiments we use apple_ocr_openai

- k=1
- k=3
- k=5

In [ ]:
run_bm25_rag_eval(
    eval_csv_path="evaluation_questions.csv",
    k=5,
)